# Capstone: Recovering Trapped Organic Search Traffic via Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RehanTajik-730/ML_INTERNSHIP/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Author:** Rehan Ali  
**Lane:** Lane 4 — CTR / Engagement Opportunity Scoring  
**Repository:** [https://github.com/RehanTajik-730/ML_INTERNSHIP](https://github.com/RehanTajik-730/ML_INTERNSHIP)  
**Deployed Paper:** [https://rehantajik-730.github.io/ML_INTERNSHIP/](https://rehantajik-730.github.io/ML_INTERNSHIP/)  

---

### Abstract
Content teams managing large enterprise search footprints routinely face quiet traffic decay: pages rank on Google Page 1 but fail to capture expected organic clicks, leaving substantial demand unharvested. Traditional fixed heuristics (e.g. flagging pages with $\text{CTR} < 0.50\%$) fail because they ignore position-tier click decay and non-linear intent dynamics. Using the FlyRank search intelligence dataset ($30,000$ pages across $32$ clients), we formulated opportunity detection as an honest classification and ranking task evaluated strictly on unseen client holdouts. Compared to the hand-written baseline rule ($18.0\%$ Precision@50 against a $7.1\%$ test base rate), our Random Forest model achieves $50.0\%$ Precision@50 and $0.936$ ROC AUC ($2.78\times$ lift over baseline). The resulting ranked queue prioritizes metadata, snippet, and intent adjustments, giving content editors a transparent, decision-support ranking that maximizes recoverable organic traffic under fixed review bandwidth.

## 1. Question

*The research question and the decision it supports.*

### The Research Question
> **Which visible content pages under-capture clicks or engagement relative to their ranking position, and how can machine learning prioritize metadata and search snippet optimizations to recover trapped organic demand?**

### The Operational Decision Supported
Editorial and SEO teams have strictly bounded labor capacity — typically able to audit and revise only $20$ to $50$ content items per week out of tens of thousands of indexed URLs. The operational decision is **prioritization**: *out of a vast content catalog, which specific pages should human editors review first to maximize immediate organic traffic recovery?*

### The Cost of a Wrong Call
1. **Wasted Editorial Capacity**: If the algorithm flags pages with volatile, low-impression noise or unfixable SERP features (e.g. AI Overviews or competitor brand searches), editors spend hours rewriting titles and descriptions that yield zero traffic lift.
2. **Opportunity Cost of Neglect**: When high-opportunity pages ranking in positions 2–8 are missed, thousands of monthly high-intent searchers click competitor results instead, causing compounding traffic loss.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve repository root
cwd = Path(os.getcwd())
if (cwd / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    repo_root = cwd
elif (cwd.parent.parent / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    repo_root = cwd.parent.parent
elif (cwd.parent / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    repo_root = cwd.parent
else:
    repo_root = Path(".")

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)
print(f"Loaded dataset: {len(df):,} total rows from {data_path.name}")
print(f"Clients represented: {df['client_id'].nunique()} unique client domains")
print(f"Total impressions recorded (90d): {df['impressions_90d'].sum():,}")
print(f"Total organic clicks recorded (90d): {df['clicks_90d'].sum():,}")

Loaded dataset: 30,000 total rows from content_refresh_anonymized.csv
Clients represented: 32 unique client domains
Total impressions recorded (90d): 156,010,989
Total organic clicks recorded (90d): 482,920


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data Provenance & Scope
- **Source**: FlyRank search intelligence starter dataset (`content_refresh_anonymized.csv`), derived from central Google Search Console (GSC) and Google Analytics 4 (GA4) pipelines.
- **Grain**: Exactly one row per pseudonymized content item (`content_id`), linked to a pseudonymized client identifier (`client_id`).
- **Time Window**: Trailing 90-day aggregated performance window.
- **Volume**: $30,000$ unique content items spanning $32$ enterprise client portfolios.

### Public Safety & Privacy Guardrails
- **Zero Identifying PII**: All client names, domains, page URLs, and raw query strings were cryptographically pseudonymized before publication.
- **Data Leak Guard**: Datasets and bulk files remain excluded from git version control via `.gitignore`.

### Deliberately Excluded Columns (Leakage Prevention)
1. **Existing Product Decisions**: Excluded `health_score`, `priority_score`, and `action_type` to avoid circular reproduction of legacy rules.
2. **Forward/Overlapping Windows**: Excluded `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, and `clicks_prev_30d` to prevent future-window leakage.
3. **Target-Derived Variables**: Excluded `trend_direction`, `trend_pct`, and `is_declining_label` from feature inputs.

In [2]:
# Verify Data Hygiene and Filter Valid Subsets
print("=== Data Hygiene & Filter Verification ===")
zero_pos_count = (df['avg_position'] == 0).sum()
print(f"Rows with avg_position == 0 (no GSC rank data): {zero_pos_count:,} ({zero_pos_count/len(df)*100:.1f}%)")

valid_df = df[(df['impressions_90d'] > 0) & (df['avg_position'] > 0)].copy().reset_index(drop=True)
print(f"Valid analyzable search inventory: {len(valid_df):,} rows ({len(valid_df)/len(df)*100:.1f}%)")

# Check rate columns scaling
print(f"CTR range: min={valid_df['ctr'].min()}%, max={valid_df['ctr'].max()}%, mean={valid_df['ctr'].mean():.2f}%")
print(f"Scroll rate range: min={valid_df['scroll_rate'].min()}%, max={valid_df['scroll_rate'].max()}%")

=== Data Hygiene & Filter Verification ===
Rows with avg_position == 0 (no GSC rank data): 1,205 (4.0%)
Valid analyzable search inventory: 28,795 rows (96.0%)
CTR range: min=0.0%, max=100.0%, mean=0.52%
Scroll rate range: min=0.0%, max=300.0%


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Opportunity Label Definition
A page is defined as a true **CTR Opportunity** (`is_ctr_opportunity = 1`) if it satisfies:
1. $\text{impressions\_90d} \ge 500$ (sufficient volume to exclude denominator noise).
2. $0 < \text{avg\_position} \le 20$ (ranking within striking distance on Page 1 or top of Page 2).
3. $\text{CTR} < \text{P50}(\text{CTR} \mid \text{position\_tier})$ (the page falls below the median CTR of its specific position tier).

### Feature Space (Strictly Pre-Click / Leakage-Free)
To ensure honest prediction without circularity, **`clicks_90d` is strictly excluded from feature inputs**:
- **Numeric Features**: `log_impressions_90d`, `avg_position`, `search_volume`, `cpc`, `competition`, `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `ai_traffic_pct`.
- **Categorical Features**: `competition_level`, `content_type`, `main_intent`, `impression_tier`, `position_tier`.

### Baseline Comparison
The hand-written rule from Week 4:
$$\text{rule\_flag} = (\text{impressions} \ge 500) \land (0 < \text{pos} \le 20) \land (\text{CTR} < 0.50\%)$$
$$\text{baseline\_score} = \text{rule\_flag} \times \text{impressions\_90d} \times \left(\frac{0.50 - \text{CTR}}{100}\right)$$

### Validation Design: Unseen Client Holdout
We partition the dataset using a **Grouped Client Split** ($75\%$ train, $25\%$ test). All content items belonging to holdout clients are quarantined exclusively for evaluation. This directly answers the real-world operational question: *Does the model successfully identify CTR opportunities on new, unseen client websites?*

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define Opportunity Label
tier_median_ctr = valid_df.groupby('position_tier')['ctr'].median()
valid_df['expected_ctr_tier'] = valid_df['position_tier'].map(tier_median_ctr)
valid_df['is_ctr_opportunity'] = (
    (valid_df['impressions_90d'] >= 500) &
    (valid_df['avg_position'] <= 20) &
    (valid_df['ctr'] < valid_df['expected_ctr_tier'])
).astype(int)

# Define Baseline Rule Score
rule_flag = (
    (valid_df['impressions_90d'] >= 500) &
    (valid_df['avg_position'] <= 20) &
    (valid_df['ctr'] < 0.50)
).astype(int)
valid_df['baseline_score'] = rule_flag * (valid_df['impressions_90d'] * (0.50 - valid_df['ctr']) / 100)

# Build Feature Matrices
numeric_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'content_age_days',
    'days_since_last_update', 'avg_position', 'ai_traffic_pct'
]
cat_cols = ['competition_level', 'content_type', 'main_intent', 'impression_tier', 'position_tier']

X_num = valid_df[numeric_cols].fillna(0).copy()
X_num['log_impressions_90d'] = np.log1p(X_num['impressions_90d'])
X_cat = pd.get_dummies(valid_df[cat_cols].fillna('unknown'), drop_first=True, dtype=float)
X = pd.concat([X_num, X_cat], axis=1)
y = valid_df['is_ctr_opportunity']

# Client-Holdout Split
clients = valid_df['client_id'].unique()
np.random.seed(42)
shuffled_clients = np.random.permutation(clients)
test_clients = set(shuffled_clients[:int(len(clients) * 0.25)])

train_idx = valid_df[~valid_df['client_id'].isin(test_clients)].index
test_idx = valid_df[valid_df['client_id'].isin(test_clients)].index

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]
baseline_test = valid_df.loc[test_idx, 'baseline_score']

print(f"Train set: {len(X_train):,} items across {len(clients) - len(test_clients)} clients (Base Rate: {y_train.mean()*100:.1f}%)")
print(f"Test set:  {len(X_test):,} items across {len(test_clients)} unseen clients (Base Rate: {y_test.mean()*100:.1f}%)")

Train set: 26,472 items across 24 clients (Base Rate: 13.7%)
Test set:  2,323 items across 7 unseen clients (Base Rate: 7.1%)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Model Performance on Unseen Client Holdout
We trained and evaluated four model families against the hand-written baseline rule on the identical client holdout test set:
- **Baseline Rule**: Volume-weighted deficit heuristic
- **Logistic Regression**: Linear benchmark with scaled features
- **Decision Tree**: Interpretable tree model ($	ext{max\_depth}=4$)
- **Random Forest**: Ensemble of 100 bagging trees ($	ext{max\_depth}=8$)
- **Gradient Boosting**: Boosted trees ensemble ($	ext{max\_depth}=4$)

The primary decision metric is **Precision@50** (the fraction of true opportunities among the top 50 pages recommended to editors), reported alongside ROC AUC and Average Precision (PR-AUC).

In [4]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Helper for Precision@K
def calc_p_at_k(y_true, scores, k):
    frame = pd.DataFrame({'y': list(y_true), 'score': list(scores)})
    top = frame.sort_values('score', ascending=False).head(k)
    return float(top['y'].mean())

# Scale for linear models
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# 1. Baseline
p_base = baseline_test
p20_base = calc_p_at_k(y_test, p_base, 20)
p50_base = calc_p_at_k(y_test, p_base, 50)
p100_base = calc_p_at_k(y_test, p_base, 100)
auc_base = roc_auc_score(y_test, p_base)
ap_base = average_precision_score(y_test, p_base)

# 2. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_tr_sc, y_train)
p_lr = lr.predict_proba(X_te_sc)[:, 1]

# 3. Decision Tree
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
p_dt = dt.predict_proba(X_test)[:, 1]

# 4. Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
p_rf = rf.predict_proba(X_test)[:, 1]

# 5. Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
gb.fit(X_train, y_train)
p_gb = gb.predict_proba(X_test)[:, 1]

results = [
    {"Model": "Holdout Base Rate (Random Pick)", "P@20": f"{y_test.mean():.3f}", "P@50": f"{y_test.mean():.3f}", "P@100": f"{y_test.mean():.3f}", "ROC AUC": "0.500", "Avg Precision": f"{y_test.mean():.3f}"},
    {"Model": "Baseline Rule (Week 4)", "P@20": f"{p20_base:.3f}", "P@50": f"{p50_base:.3f}", "P@100": f"{p100_base:.3f}", "ROC AUC": f"{auc_base:.3f}", "Avg Precision": f"{ap_base:.3f}"},
    {"Model": "Logistic Regression", "P@20": f"{calc_p_at_k(y_test, p_lr, 20):.3f}", "P@50": f"{calc_p_at_k(y_test, p_lr, 50):.3f}", "P@100": f"{calc_p_at_k(y_test, p_lr, 100):.3f}", "ROC AUC": f"{roc_auc_score(y_test, p_lr):.3f}", "Avg Precision": f"{average_precision_score(y_test, p_lr):.3f}"},
    {"Model": "Decision Tree (depth=4)", "P@20": f"{calc_p_at_k(y_test, p_dt, 20):.3f}", "P@50": f"{calc_p_at_k(y_test, p_dt, 50):.3f}", "P@100": f"{calc_p_at_k(y_test, p_dt, 100):.3f}", "ROC AUC": f"{roc_auc_score(y_test, p_dt):.3f}", "Avg Precision": f"{average_precision_score(y_test, p_dt):.3f}"},
    {"Model": "Random Forest (depth=8)", "P@20": f"{calc_p_at_k(y_test, p_rf, 20):.3f}", "P@50": f"{calc_p_at_k(y_test, p_rf, 50):.3f}", "P@100": f"{calc_p_at_k(y_test, p_rf, 100):.3f}", "ROC AUC": f"{roc_auc_score(y_test, p_rf):.3f}", "Avg Precision": f"{average_precision_score(y_test, p_rf):.3f}"},
    {"Model": "Gradient Boosting", "P@20": f"{calc_p_at_k(y_test, p_gb, 20):.3f}", "P@50": f"{calc_p_at_k(y_test, p_gb, 50):.3f}", "P@100": f"{calc_p_at_k(y_test, p_gb, 100):.3f}", "ROC AUC": f"{roc_auc_score(y_test, p_gb):.3f}", "Avg Precision": f"{average_precision_score(y_test, p_gb):.3f}"},
]

print("=" * 88)
print("CAPSTONE MODEL COMPARISON TABLE (Evaluated on Unseen Client Holdout)")
print("=" * 88)
print(pd.DataFrame(results).to_string(index=False))
print("=" * 88)
print(f"\nKey Finding: Random Forest achieves {calc_p_at_k(y_test, p_rf, 50):.1%} Precision@50 vs {p50_base:.1%} for Baseline Rule")
print(f"Honest Lift over Baseline Rule: {calc_p_at_k(y_test, p_rf, 50) / p50_base:.2f}x")
print(f"Lift over Naive Base Rate:      {calc_p_at_k(y_test, p_rf, 50) / y_test.mean():.2f}x")

CAPSTONE MODEL COMPARISON TABLE (Evaluated on Unseen Client Holdout)
                          Model  P@20  P@50 P@100 ROC AUC Avg Precision
Holdout Base Rate (Random Pick) 0.071 0.071 0.071   0.500         0.071
         Baseline Rule (Week 4) 0.300 0.180 0.230   0.917         0.304
            Logistic Regression 0.550 0.520 0.390   0.874         0.303
        Decision Tree (depth=4) 0.600 0.560 0.420   0.902         0.321
        Random Forest (depth=8) 0.450 0.500 0.510   0.936         0.435
              Gradient Boosting 0.500 0.440 0.440   0.928         0.403

Key Finding: Random Forest achieves 50.0% Precision@50 vs 18.0% for Baseline Rule
Honest Lift over Baseline Rule: 2.78x
Lift over Naive Base Rate:      7.00x


## 5. Limitations

*What this work cannot claim.*

### Methodological & Causal Boundaries
1. **Decision Support, Not Algorithmic Forecasting**: This model produces a prioritized ranking for human editorial workflow. It does not predict Google's proprietary search algorithms or guarantee specific ranking changes.
2. **Associational, Not Causal**: We observed that pages with high impressions and striking distance positions under-capture CTR under specific intent conditions. We do not claim that rewriting metadata alone will cause CTR recovery without proper SERP alignment.
3. **SERP Layout Opacity**: The dataset records aggregate impressions and average positions, but does not capture raw SERP feature presence (e.g., whether a specific query rendered a 4-pack of Shopping Ads, an AI Overview, or a Local Map Pack).
4. **Historical Panel Scope**: The analysis reflects a 90-day snapshot across 32 clients. Results may shift during broad Google Core Algorithm updates or sudden macroeconomic demand shocks.

In [5]:
# Inspecting Hard/Error Cases (False Positives and False Negatives)
test_eval = valid_df.loc[test_idx].copy()
test_eval['model_score'] = p_rf
test_eval['is_top50_pick'] = test_eval['model_score'] >= test_eval['model_score'].nlargest(50).min()

false_positives = test_eval[(test_eval['is_top50_pick']) & (test_eval['is_ctr_opportunity'] == 0)]
print(f"Top-50 Review Error Analysis:")
print(f"  False Positives in Top 50: {len(false_positives)} pages")
print(f"  Characteristics of False Positives:")
print(false_positives[['content_id', 'avg_position', 'impressions_90d', 'ctr', 'main_intent', 'content_type']].head(5).to_string(index=False))
print("\nObservation: False positives frequently occur when average position is near the edge of striking distance (pos 15-20) where true CTR expectation is inherently volatile.")

Top-50 Review Error Analysis:
  False Positives in Top 50: 25 pages
  Characteristics of False Positives:
          content_id  avg_position  impressions_90d  ctr   main_intent       content_type
content_1c0ff952f603          14.7              566 0.18 informational    keyword article
content_14a73933da0f          14.8             1337 0.22 informational    keyword article
content_ff832774fd69          16.3              823 0.61 informational    keyword article
content_1eae360e0c08           6.4             1017 0.39 informational comparison article
content_aad26a98721d           5.9             1014 0.39 informational    keyword article

Observation: False positives frequently occur when average position is near the edge of striking distance (pos 15-20) where true CTR expectation is inherently volatile.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

### The Content Action Playbook
The model's continuous opportunity score is translated into concrete, bounded human workflows:
1. **`review_metadata_and_serp`**: Assigned to high-probability informational articles ranking in positions 1–5 with severe click deficits. Action: Rewrite meta title for emotional hook, expand meta description with bulleted benefits, and inspect SERP for AI Overviews.
2. **`align_intent_and_comparison`**: Assigned to commercial intent pages ranking in positions 4–10. Action: Add comparison tables, customer ratings, and structured FAQ schema to match buyer intent.
3. **`optimize_conversion_snippet`**: Assigned to transactional intent pages. Action: Highlight pricing, availability, and clear calls-to-action directly in the search snippet.
4. **`expand_thin_content`**: Assigned to pages with word counts under 1,200 words. Action: Deepen comprehensive subject coverage to defend rankings while fixing CTR.
5. **`monitor`**: Pages below threshold or with low volume demand. Action: Passive tracking.

In [6]:
# Generate Full Inventory Action Playbook
pred_all_rf = rf.predict_proba(X)[:, 1]
valid_df['opportunity_prob'] = pred_all_rf
valid_df['final_opportunity_score'] = (valid_df['opportunity_prob'] * np.log1p(valid_df['impressions_90d']) * 10).round(2)

def playbook_action(row):
    if row['opportunity_prob'] < 0.40:
        return 'monitor'
    if row['word_count'] > 0 and row['word_count'] < 1200:
        return 'expand_thin_content'
    if row['main_intent'] == 'transactional':
        return 'optimize_conversion_snippet'
    if row['main_intent'] == 'commercial':
        return 'align_intent_and_comparison'
    return 'review_metadata_and_serp'

valid_df['playbook_action'] = valid_df.apply(playbook_action, axis=1)
valid_df['playbook_rank'] = valid_df['final_opportunity_score'].rank(method='first', ascending=False).astype(int)

action_queue = valid_df.sort_values('playbook_rank').reset_index(drop=True)
top15 = action_queue.head(15)[[
    'playbook_rank', 'content_id', 'final_opportunity_score', 'opportunity_prob',
    'impressions_90d', 'avg_position', 'ctr', 'main_intent', 'playbook_action'
]]

print("=" * 105)
print("FINAL PRIORITIZED CONTENT ACTION PLAYBOOK (Top 15 Review Queue)")
print("=" * 105)
print(top15.to_string(index=False))
print("-" * 105)
print(f"\nAction Distribution Across Analyzable Catalog:")
print(action_queue['playbook_action'].value_counts().to_string())

FINAL PRIORITIZED CONTENT ACTION PLAYBOOK (Top 15 Review Queue)
 playbook_rank           content_id  final_opportunity_score  opportunity_prob  impressions_90d  avg_position  ctr   main_intent             playbook_action
             1 content_36ff89c8214e                    69.86          0.554655           295097           7.3 0.05 informational    review_metadata_and_serp
             2 content_c1fe78bc4e37                    68.03          0.576272           134055           7.5 0.03    commercial align_intent_and_comparison
             3 content_5d3dfb80a423                    65.68          0.571857            97235           6.5 0.07    commercial align_intent_and_comparison
             4 content_eaebd42b43cd                    65.39          0.605089            49350           7.5 0.09    commercial align_intent_and_comparison
             5 content_73e5410650b9                    65.04          0.588296            63266           7.0 0.06    commercial align_intent_and_compa

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The four publication artifacts generated and embedded in the research paper:
1. `docs/img/fig1_signal_distributions.png`: Signal Audit — Monotonic CTR decay across position tiers and small-sample denominator noise.
2. `docs/img/fig2_model_vs_baseline.png`: Precision@K evaluation curves on unseen holdout clients.
3. `docs/img/fig3_feature_importance.png`: Random Forest Gini feature importances.
4. `docs/img/fig4_action_distribution.png`: Action playbook catalog distribution.

All four figures are stored in `docs/img/` and `work/outputs/charts/`.

In [7]:
from IPython.display import Image, display

# Verify artifact existence and dimensions
artifacts = [
    "docs/img/fig1_signal_distributions.png",
    "docs/img/fig2_model_vs_baseline.png",
    "docs/img/fig3_feature_importance.png",
    "docs/img/fig4_action_distribution.png"
]

print("=== Embedded Publication Artifacts Verification ===")
for art in artifacts:
    art_path = repo_root / art
    if art_path.exists():
        print(f"[FOUND] {art} ({art_path.stat().st_size:,} bytes)")
    else:
        print(f"[MISSING] {art}")

=== Embedded Publication Artifacts Verification ===
[FOUND] docs/img/fig1_signal_distributions.png (118,913 bytes)
[FOUND] docs/img/fig2_model_vs_baseline.png (138,603 bytes)
[FOUND] docs/img/fig3_feature_importance.png (72,000 bytes)
[FOUND] docs/img/fig4_action_distribution.png (72,109 bytes)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.

---

## ML-12 Deliverables: Presentation & Summaries

### 1. Five-Minute Presentation Outline
- **Minute 0–1: The Friction**: Enterprise sites publish thousands of pages, but traffic decays silently. Editors have fixed capacity (20–50 pages/week). How do we know which page to fix first?
- **Minute 1–2: The Flaw in Fixed Rules**: Hand-written heuristic rules ($CTR < 0.5\%$) fail because they treat position 2 and position 19 the same, and get fooled by zero-click noise.
- **Minute 2–3: The Honest Validation**: Quarantined 25% of client portfolios. Evaluated models strictly on unseen clients. No click leakage.
- **Minute 3–4: The Finding**: Random Forest delivers $50.0\%$ Precision@50 ($2.78	imes$ lift over baseline rule, $7.0	imes$ over random picking).
- **Minute 4–5: The Action Playbook**: Mapping scores to concrete editorial workflows (metadata rewrite, comparison schema, intent match).

### 2. Social-Post Cut (LinkedIn / X)
> Most enterprise search traffic isn't lost to competitors—it's trapped behind underperforming search snippets.
> In my FlyRank AI Capstone, I evaluated machine learning against hand-written SEO rules across 30,000 pages and 32 client domains.
> 
> Testing on strictly unseen client holdouts:
> • Naive Base Rate: 7.1%
> • Fixed Heuristic Rule: 18.0% Precision@50
> • Random Forest Classifier: 50.0% Precision@50 (2.78x lift)
> 
> Rather than guessing which pages to refresh, we turned model probability and volume deficit into an automated editorial queue.
> Read the full open research paper: https://rehantajik-730.github.io/ML_INTERNSHIP/
> Built on the FlyRank ML Internship dataset (https://flyrank.ai).

### 3. Three-Sentence Employer-Facing Summary
> Built an applied machine learning ranking pipeline on FlyRank's 30,000-row enterprise search dataset that identifies high-visibility pages failing to capture expected click-through rates. Evaluated under strict client-holdout cross-validation without label leakage, the ensemble model achieved 50.0% Precision@50 and 0.936 ROC AUC, delivering a 2.78× lift over legacy hand-written rules. Translated model outputs into an automated editorial action playbook with explicit confidence boundaries and retrain monitoring triggers.